In [1]:
import os
import sys
import threading
from datetime import datetime

cwd = os.getcwd()

sys.path.append("/data/alop/eye_transformer/")
from utils.tokenizer_aligner import TokenizerAligner
from transformers import AutoTokenizer

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorWithPadding,
    BatchEncoding,
)

from functools import partial
from datasets import load_dataset, Dataset

/home/alop/miniconda3/envs/cache_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PRIMERO LO INTENTO CON DATASETS

In [2]:
tokenizer_name = "t5-small"
tokenizer_fix = AutoTokenizer.from_pretrained(
    tokenizer_name, cache_dir="./cache/models", model_max_length=2048
)

# tokenizer = T5Tokenizer.from_pretrained('t5-small')
tokenizer_name = "meta-llama/Meta-Llama-3-8B"
tokenizer_model = AutoTokenizer.from_pretrained(tokenizer_name)
tokenizer_model.add_special_tokens({"pad_token": "[PAD]"})
dataset = "timdettmers/openassistant-guanaco"
data = load_dataset(dataset, split="train[:2%]")
print(data)


def tokenize_function(examples, tokenizer):
    tokenized_examples = tokenizer(
        examples["text"], padding=True, truncation=True, add_special_tokens=True
    )
    return BatchEncoding(tokenized_examples)


# def tokenize_function(example, tokenizer):
#     return tokenizer(example["text"], padding=True, truncation=True, add_special_tokens=True)


def detokenize_function(example, tokenizer):
    return {"text": tokenizer.decode(example["input_ids"], skip_special_tokens=True)}


batch_size = 2
# def convert_to_batch_encoding(example):
#     return BatchEncoding(example)


class BatchEncodingDataset:
    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, index):
        item = self.dataset[index]
        return BatchEncoding(item)

    def __len__(self):
        return len(self.dataset)


# Convert the tokenized dataset to our custom class

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['text'],
    num_rows: 197
})


In [3]:
function_tokenize = partial(tokenize_function, tokenizer=tokenizer_model)
data_tok_model = data.map(function_tokenize, batched=True, batch_size=batch_size)

# data_tok_model = data_tok_model.map(convert_to_batch_encoding, batched=False)
# data_tok_model.set_format(columns=['input_ids', 'attention_mask'])
function_detokenize = partial(detokenize_function, tokenizer=tokenizer_model)
data_detok_model = data_tok_model.map(function_detokenize, batched=False)
# remove input_ids adn attention_mask columns
data_detok_model.set_format(columns=["text"])
function_tokenize = partial(tokenize_function, tokenizer=tokenizer_fix)
data_tok_fix = data_detok_model.map(
    function_tokenize, batched=True, batch_size=batch_size
)
# data_tok_fix.set_format(columns=['input_ids', 'attention_mask'])
data_tok_fix = BatchEncodingDataset(data_tok_fix)
data_tok_model = BatchEncodingDataset(data_tok_model)

Map: 100%|██████████| 197/197 [00:00<00:00, 345.31 examples/s]


In [4]:
print(len(data_tok_fix))
print(len(data_detok_model))
print(len(data_tok_model))
print(data_tok_fix[0])
print(data_detok_model[0])
print(data_tok_model[0])
print(type(data_tok_fix[0]))
print(type(data_detok_model[0]))
print(type(data_tok_model[0]))

197
197
197
{'text': '### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant: "Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the employer has little incentive to increase wages or provide better working conditions.\n\nRecent research has identified potential monopsonies in industries such as retail and fast food, where a few large companies control a significant portion of the market (Bivens & Mishel, 2013). In these industries, workers often face low wages, limited benefits, and reduced b

In [5]:
example = BatchEncoding(data_tok_fix[0])
print(type(example))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [6]:
text_tokenized = tokenizer_model(
    ["hello world", "Example this is sooo long hehe"],
    padding=True,
    truncation=True,
    add_special_tokens=True,
)
print(type(text_tokenized))
print(text_tokenized)
print(type(data_tok_model[0]))
print(text_tokenized.is_fast)
print(data_tok_model[0].is_fast)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


<class 'transformers.tokenization_utils_base.BatchEncoding'>
{'input_ids': [[128000, 15339, 1917, 128256, 128256, 128256, 128256, 128256, 128256], [128000, 13617, 420, 374, 779, 2689, 1317, 568, 383]], 'attention_mask': [[1, 1, 1, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1]]}
<class 'transformers.tokenization_utils_base.BatchEncoding'>
True
False


In [7]:
for i in range(len(data_tok_fix)):
    token_idx_mapped = TokenizerAligner().align_tokens(
        data_detok_model[0]["text"],
        BatchEncoding(data_tok_model[0]),
        BatchEncoding(data_tok_fix[0]),
    )

ValueError: word_ids() is not available when using non-fast tokenizers (e.g. instance of a `XxxTokenizerFast` class).

Para poder usar los metodos de word_ids() y word_to_chars(), para luego hacer tokenizer alignment, necesito tener una clase (transformers.tokenization_utils_base.BatchEncoding) y que sea FAST. 
Esto cuando tokenizas un solo texto estilo:
    text_tokenized = tokenizer_model(text, padding=True, truncation=True, add_special_tokens=True)
Te la devuelve y le puedes aplicar estos metodos. 
El problema es que cuando he intentando tokenizar con el map en un dataset (necesito batches) no me ha funcionado. 
He intenando crear esta clase a partir del dic con BatchEncoding() de varias maneras (map, clase particular... etz)
Pero no consigo que sea FAST entonces no puedo usar los metodos. 

La siguiente opción es intentar tokenizar sin el map del dataset pero varios textos a la vez (para poder hacer padding dle batch)
    text_tokenized = tokenizer_model([text1, text2], padding=True, truncation=True, add_special_tokens=True)    
En principio asi funciona:
    text_tokenized.word_ids()
    text_tokenized.word_to_chars()
Pero me estoy encontrnando que me devuelve solo del primero de los textos. Haciendo bucles for luego parece solucionarse. 

In [8]:
text_tokenized_uno = tokenizer_model(
    "hello world", padding=True, truncation=True, add_special_tokens=True
)
text_tokenized_dos = tokenizer_model(
    ["hello world", "Example this is sooo long hehe"],
    padding=True,
    truncation=True,
    add_special_tokens=True,
)
print(isinstance(text_tokenized_uno["input_ids"][0], list))
print(isinstance(text_tokenized_dos["input_ids"][0], list))

False
True


In [9]:
print(text_tokenized_uno.word_ids())
print(text_tokenized_uno.word_ids())
# iterando sin los () puedo sacar los word_ids de los dos
for i in range(len(text_tokenized_dos["input_ids"])):
    print(text_tokenized_dos[i].word_ids)

[None, 0, 1]
[None, 0, 1]
[None, 0, 1, None, None, None, None, None, None]
[None, 0, 1, 2, 3, 3, 4, 5, 5]


In [10]:
examples = ["hello world", "Example this is sooo long hehe"]
text_tokenized_alls = tokenizer_model(
    examples, padding=True, truncation=True, add_special_tokens=True
)
test_tokenize_list = [
    tokenizer_model(example, padding=True, truncation=True, add_special_tokens=True)
    for example in examples
]
word_ids_all = [
    text_tokenized_alls[i].word_ids
    for i in range(len(text_tokenized_alls["input_ids"]))
]
word_ids_list = [text_tokenized.word_ids() for text_tokenized in test_tokenize_list]
print(word_ids_all)
print(word_ids_list)
steps_all = [
    [
        (text_tokenized_alls[i].word_to_chars(j))
        for j in range(
            max([x if x is not None else 0 for x in text_tokenized_alls[i].word_ids])
        )
    ]
    for i in range(len(text_tokenized_alls["input_ids"]))
]
steps_list = [
    [
        (text_tokenized.word_to_chars(i))
        for i in range(
            max([x if x is not None else 0 for x in text_tokenized.word_ids()])
        )
    ]
    for text_tokenized in test_tokenize_list
]
words_all = [
    [
        [
            examples[i][
                text_tokenized_alls[i].word_to_chars(j)[0] : text_tokenized_alls[
                    i
                ].word_to_chars(j)[1]
            ]
            .strip()
            .lower()
        ]
        for j in range(
            1
            + max([x if x is not None else 0 for x in text_tokenized_alls[i].word_ids])
        )
    ]
    for i in range(len(text_tokenized_alls["input_ids"]))
]

print(steps_all)
print(steps_list)
print(words_all)

[[None, 0, 1, None, None, None, None, None, None], [None, 0, 1, 2, 3, 3, 4, 5, 5]]
[[None, 0, 1], [None, 0, 1, 2, 3, 3, 4, 5, 5]]
[[(0, 5)], [(0, 7), (7, 12), (12, 15), (15, 20), (20, 25)]]
[[CharSpan(start=0, end=5)], [CharSpan(start=0, end=7), CharSpan(start=7, end=12), CharSpan(start=12, end=15), CharSpan(start=15, end=20), CharSpan(start=20, end=25)]]
[[['hello'], ['world']], [['example'], ['this'], ['is'], ['sooo'], ['long'], ['hehe']]]


In [11]:
def _text_to_words_batch(texts, text_tokenized_alls):
    words_all = []
    for i in range(len(text_tokenized_alls["input_ids"])):
        words = []
        for j in range(
            1
            + max([x if x is not None else 0 for x in text_tokenized_alls[i].word_ids])
        ):
            chars = text_tokenized_alls[i].word_to_chars(j)
            words.append([texts[i][chars[0] : chars[1]].strip().lower()])
        words_all.append(words)
    return words_all


words_all = _text_to_words_batch(examples, text_tokenized_alls)
print(words_all)

[[['hello'], ['world']], [['example'], ['this'], ['is'], ['sooo'], ['long'], ['hehe']]]


Cuando tratas una clase de batchcoding que viene de un texto o mas de uno, hay diferencias en los metodos que puedes aplicar.

Por ejemplo la unica manera de sacar los words_ids() y word_to_chars() cuando es mas de un texto es hacer text_tokenized[i].word_ids, sin la ()

Por lo que al final cambio la clase de TokenizerAligner para poder iterar sobre batches

In [12]:
examples = ["hello world", "Example this is sooo long hehe"]
text_tokenized_all = tokenizer_model(
    examples, padding=True, truncation=True, add_special_tokens=True
)
word_ids_all = [
    text_tokenized_all[i].word_ids for i in range(len(text_tokenized_all["input_ids"]))
]
print(word_ids_all)
words_all = [
    [
        [
            examples[i][
                text_tokenized_all[i].word_to_chars(j)[0] : text_tokenized_all[
                    i
                ].word_to_chars(j)[1]
            ]
            .strip()
            .lower()
        ]
        for j in range(
            1 + max([x if x is not None else 0 for x in text_tokenized_all[i].word_ids])
        )
    ]
    for i in range(len(text_tokenized_all["input_ids"]))
]
print(words_all)

[[None, 0, 1, None, None, None, None, None, None], [None, 0, 1, 2, 3, 3, 4, 5, 5]]
[[['hello'], ['world']], [['example'], ['this'], ['is'], ['sooo'], ['long'], ['hehe']]]


Ahora que funciona para batches, vemos como conseguir tokenizar y destokenizar antes

In [13]:
tokenizer_name = "t5-small"
tokenizer_fix = AutoTokenizer.from_pretrained(
    tokenizer_name, cache_dir="./cache/models", model_max_length=2048
)

# tokenizer = T5Tokenizer.from_pretrained('t5-small')
tokenizer_name = "meta-llama/Meta-Llama-3-8B"
tokenizer_model = AutoTokenizer.from_pretrained(tokenizer_name)
tokenizer_model.add_special_tokens({"pad_token": "[PAD]"})
dataset = "timdettmers/openassistant-guanaco"
data = load_dataset(dataset, split="train[:1%]")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Repo card metadata block was not found. Setting CardData to empty.


In [8]:
text_init = data["text"][6]
# text_init = "this is insano, want to die"
print(text_init)
tokenized_text_model = tokenizer_model(
    text_init, padding=True, truncation=True, add_special_tokens=True
)
texts = tokenizer_model.decode(
    tokenized_text_model["input_ids"], skip_special_tokens=True
)
tokenized_text_fix = tokenizer_fix(
    texts, padding=True, truncation=True, add_special_tokens=True
)
mapped = TokenizerAligner().align_tokens(
    texts, tokenized_text_model, tokenized_text_fix
)
tokens_str_model = tokenizer_model.tokenize(
    texts, padding=True, truncation=True, add_special_tokens=True
)
print(tokens_str_model)
print(tokenized_text_model["input_ids"])
tokens_str_fix = tokenizer_fix.tokenize(
    texts, padding=True, truncation=True, add_special_tokens=True
)
print(tokens_str_fix)
print(tokenized_text_fix["input_ids"])
token_idx_str_mapped = TokenizerAligner.map_tokens_to_str(
    mapped, tokens_str_model, tokens_str_fix
)
print("--------------")
print(mapped)
print("--------------")
for i in token_idx_str_mapped:
    print(i)

### Human: Can you give me an example of a python script that opens an api point and serves a string?### Assistant: Sure! Here's an example Python script that uses the Flask web framework to create a simple API endpoint that serves a string:

``` 
from flask import Flask

app = Flask(__name__)

@app.route('/')
def hello_world():
    return 'Hello, world!'

if __name__ == '__main__':
    app.run()

``` 

In this script, we first import the Flask class from the flask module. Then we create a new instance of the Flask class, using the __name__ variable to specify the name of the application.
\
Next, we define a new route using the @app.route() decorator. This decorator tells Flask to map requests to the root URL ("/") to the hello_world() function.
\
Finally, we use the if __name__ == '__main__': block to start the Flask application when the script is executed. By default, the application will run on port 5000.
\
You can run this script and test the API by opening a web browser and naviga

In [16]:
tokenized_text_model = tokenizer_model(
    data["text"][1:5], padding=True, truncation=True, add_special_tokens=True
)
texts = []
for i in range(len(tokenized_text_model["input_ids"])):
    texts.append(
        tokenizer_model.decode(
            tokenized_text_model["input_ids"][i], skip_special_tokens=True
        )
    )
tokenized_text_fix = tokenizer_fix(
    texts, padding=True, truncation=True, add_special_tokens=True
)
mapped = TokenizerAligner().align_tokens(
    texts, tokenized_text_model, tokenized_text_fix
)

FUNCIONA CON BATCHES